In [1]:
!pip install azure-storage-blob

In [2]:
#接続文字列（Azure Portalからコピーしたもの）
import os
CONNECTION_STRING = os.environ.get("AZURE_STORAGE_CONNECTION_STRING", "")

In [3]:
from azure.storage.blob import BlobServiceClient
from azure.core.exceptions import ResourceExistsError

# 設定
CONTAINER_NAME = "pdf-output"

# クライアント作成
client = BlobServiceClient.from_connection_string(CONNECTION_STRING)

# コンテナ作成
try:
    container_client = client.create_container(CONTAINER_NAME)
    print("コンテナを作成しました")
except ResourceExistsError:
    print("コンテナは既に存在します（スキップ）")

コンテナは既に存在します（スキップ）


In [4]:
# アップロードするコンテナとファイル名を指定
container_name = "pdf-output"
blob_name = "test.txt"
upload_data = "これはテストファイルです"

# Blobクライアント作成
blob_client = client.get_blob_client(
    container=container_name,
    blob=blob_name
)

# アップロード実行
# 既に存在する場合はスキップ
if not blob_client.exists():
    blob_client.upload_blob(upload_data)
    print(f"{blob_name} をアップロードしました")
else:
    print(f"{blob_name} は既に存在します（スキップ）")

test.txt は既に存在します（スキップ）


In [5]:
#コンテナ内のファイル一覧を取得
container_client = client.get_container_client(container_name)

print("コンテナ内のファイル一覧：")
for blob in container_client.list_blobs():
    print(f" -{blob.name}")

コンテナ内のファイル一覧：
 -report_2026-03-12.pdf
 -report_2026-03-13.pdf
 -report_2026-03-14.pdf
 -test.txt


In [6]:
#Download実行
download_client = client.get_blob_client(
    container=container_name,
    blob=blob_name
)

data = download_client.download_blob().readall()

#バイト列を文字列に変換
print("ファイルの中身：")
print(data.decode("utf-8"))

ファイルの中身：
これはテストファイルです


In [7]:
!pip install reportlab

In [8]:
#まずはシンプルなPDFを作成する
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4

#PDFファイル名
filename = "test_output.pdf"

#キャンバス作成
c = canvas.Canvas(filename, pagesize=A4)

#用紙サイズの取得
width, height = A4 #width=595, height=842(pt単位)

#テキストを描画(左から100, 下から750の位置に描画という意味)
c.drawString(100, 750, "Hello, Reportlab!")

#PDFを保存
c.save()

print(f"{filename}を作成しました")

test_output.pdfを作成しました


In [9]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
import os

#Windowsの標準フォントを登録
font_path = "C:/Windows/Fonts/msgothic.ttc"
pdfmetrics.registerFont(TTFont("MSGothic", font_path))

#PDF作成
filename = "test_japanese.pdf"
c = canvas.Canvas(filename, pagesize=A4)
width, height = A4

#フォントを指定してテキスト描画
c.setFont("MSGothic", 16)
c.drawString(100, 750, "日本語のテスト")
c.setFont("MSGothic", 12)
c.drawString(100, 720, "ポートフォリオ③PDF自動生成")

c.save()
print(f"{filename}を作成しました")

test_japanese.pdfを作成しました


In [10]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from azure.storage.blob import BlobServiceClient
import os

# ----データリスト（複数Report）----
reports = [
    {"title": "自動生成レポート", "date": "2026-03-12", "author": "S_Tasaki", "content":  "1号機のデータです。"},
    {"title": "自動生成レポート", "date": "2026-03-13", "author": "S_Tasaki", "content":  "2号機のデータです。"},
    {"title": "自動生成レポート", "date": "2026-03-14", "author": "S_Tasaki", "content":  "3号機のデータです。"},
]

# ----Loopで一括生成・Upload----
for report_data in reports:
    #PDF作成
    local_filename = "report_temp.pdf"
    c  = canvas.Canvas(local_filename, pagesize=A4)
    width, height = A4

    c.setFont("MSGothic", 20)
    c.drawString(100, 780, report_data["title"])
    c.setFont("MSGothic", 12)
    c.drawString(100, 740, f"作成日:{report_data['date']}")
    c.drawString(100, 720, f"作成者:{report_data['author']}")
    c.drawString(100, 680, report_data["content"])
    c.save()
    
    # ----Blob Storageにアップロード ----
    blob_name = f"report_{report_data['date']}.pdf"
    blob_client = client.get_blob_client(
        container=CONTAINER_NAME,
        blob=blob_name
    )
    with open(local_filename, "rb") as f:
        blob_client.upload_blob(f, overwrite=True)
    
    # ----ローカルの一時ファイルを削除 ----
    os.remove(local_filename)
    print(f"完了:{blob_name}")

完了:report_2026-03-12.pdf
完了:report_2026-03-13.pdf
完了:report_2026-03-14.pdf


In [11]:
container_client = client.get_container_client(CONTAINER_NAME)

print("コンテナ内のファイル一覧：")
for blob in container_client.list_blobs():
    print(f"  - {blob.name}")

コンテナ内のファイル一覧：
  - report_2026-03-12.pdf
  - report_2026-03-13.pdf
  - report_2026-03-14.pdf
  - test.txt
